<a href="https://colab.research.google.com/github/Alaa-f-Abdalaal/CNN-/blob/main/Search_Engine2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [47]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import requests

# NLTK downloads
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [48]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    text = re.sub(r'[^\w\s]', '', text.lower())
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and t.isalpha()]
    return tokens


In [49]:
book_links = [
    "https://www.gutenberg.org/files/1342/1342-0.txt",  # Pride and Prejudice
    "https://www.gutenberg.org/files/11/11-0.txt",      # Alice in Wonderland
    "https://www.gutenberg.org/files/98/98-0.txt"       # A Tale of Two Cities
]

def load_gutenberg_book(url):
    r = requests.get(url)
    r.encoding = "utf-8"
    return r.text

def clean_gutenberg_text(text):
    start = "*** START OF THIS PROJECT GUTENBERG EBOOK"
    end = "*** END OF THIS PROJECT GUTENBERG EBOOK"
    if start in text and end in text:
        text = text.split(start)[1].split(end)[0]
    return text

docs = []
for link in book_links:
    raw_text = load_gutenberg_book(link)
    clean_text = clean_gutenberg_text(raw_text)
    docs.append(clean_text)


In [50]:
def build_inverted_index(docs):
    inverted_index = {}  # term -> set of doc_ids
    for doc_id, doc in enumerate(docs):
        tokens = preprocess(doc)
        for token in tokens:
            if token not in inverted_index:
                inverted_index[token] = set()
            inverted_index[token].add(doc_id)
    return inverted_index

inverted_index = build_inverted_index(docs)

def boolean_search(query, inverted_index):
    q_tokens = preprocess(query)
    if not q_tokens:
        return set()
    result = inverted_index.get(q_tokens[0], set())
    for token in q_tokens[1:]:
        result = result & inverted_index.get(token, set())  # AND query
    return result


In [51]:
def edit_distance(s1, s2):
    m, n = len(s1), len(s2)
    dp = [[0]*(n+1) for _ in range(m+1)]

    for i in range(m+1):
        dp[i][0] = i
    for j in range(n+1):
        dp[0][j] = j

    for i in range(1,m+1):
        for j in range(1,n+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j-1], dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def fuzzy_search(query, inverted_index, max_dist=2):
    q_tokens = preprocess(query)
    matched_docs = set()
    for term in inverted_index.keys():
        for qt in q_tokens:
            if edit_distance(term, qt) <= max_dist:
                matched_docs.update(inverted_index[term])
    return matched_docs


In [52]:
def soundex(word):
    word = word.upper()
    codes = {"B": "1","F":"1","P":"1","V":"1",
             "C":"2","G":"2","J":"2","K":"2","Q":"2","S":"2","X":"2","Z":"2",
             "D":"3","T":"3",
             "L":"4",
             "M":"5","N":"5",
             "R":"6"}
    sound = word[0]
    for char in word[1:]:
        code = codes.get(char, "0")
        if code != sound[-1]:
            sound += code
    sound = sound.replace("0","")
    return (sound + "000")[:4]

def soundex_search(query, inverted_index):
    q_tokens = preprocess(query)
    matched_docs = set()
    query_sdx = [soundex(q) for q in q_tokens]
    for term in inverted_index.keys():
        term_sdx = soundex(term)
        if term_sdx in query_sdx:
            matched_docs.update(inverted_index[term])
    return matched_docs


In [53]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np


In [54]:
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(docs)  # كل صف = Document، كل عمود = term
terms = vectorizer.get_feature_names_out()


In [55]:
def rank_documents(query, tfidf_matrix, vectorizer, docs_list):
    q_tokens = preprocess(query)
    query_vec = np.zeros((1, len(vectorizer.get_feature_names_out())))

    for token in q_tokens:
        if token in vectorizer.vocabulary_:
            idx = vectorizer.vocabulary_[token]
            query_vec[0, idx] = 1  # weight=1 لكل كلمة في query

    # تحويل tfidf_matrix لـ numpy array إذا كانت sparse
    if hasattr(tfidf_matrix, "toarray"):
        tfidf_dense = tfidf_matrix.toarray()
    else:
        tfidf_dense = tfidf_matrix

    # Cosine similarity (dot product)
    scores = np.dot(tfidf_dense, query_vec.T).flatten()  # 1D array

    # ترتيب الـ documents حسب الـ score
    ranked_idx = np.argsort(scores)[::-1]  # من الأعلى للأقل
    ranked_docs = [(i, docs_list[i], scores[i]) for i in ranked_idx if scores[i] > 0]
    return ranked_docs


In [ ]:
while True:
    query = input("\nSearch: ").strip()
    if query.lower() == "quit":
        break

    # Boolean Search
    bool_docs = boolean_search(query, inverted_index)
    bool_ranked = rank_documents(query, tfidf_matrix, vectorizer, docs)
    bool_ranked = [d for d in bool_ranked if d[0] in bool_docs]

    # Fuzzy Search
    fuzzy_docs = fuzzy_search(query, inverted_index)
    fuzzy_ranked = rank_documents(query, tfidf_matrix, vectorizer, docs)
    fuzzy_ranked = [d for d in fuzzy_ranked if d[0] in fuzzy_docs]

    # Soundex Search
    soundex_docs = soundex_search(query, inverted_index)
    soundex_ranked = rank_documents(query, tfidf_matrix, vectorizer, docs)
    soundex_ranked = [d for d in soundex_ranked if d[0] in soundex_docs]

    # عرض النتائج
    print("\n=== Boolean Search ===")
    for doc_id, text, score in bool_ranked[:5]:
        print(f"Document [{doc_id}], Score: {score:.4f}, Link: {book_links[doc_id]}")

    print("\n=== Edit_Distance ===")
    for doc_id, text, score in fuzzy_ranked[:5]:
        print(f"Document [{doc_id}], Score: {score:.4f}, Link: {book_links[doc_id]}")

    print("\n=== Soundex Search ===")
    for doc_id, text, score in soundex_ranked[:5]:
        print(f"Document [{doc_id}], Score: {score:.4f}, Link: {book_links[doc_id]}")



Search: half whisper

=== Boolean Search ===
Document [2], Score: 0.0383, Link: https://www.gutenberg.org/files/98/98-0.txt
Document [0], Score: 0.0309, Link: https://www.gutenberg.org/files/1342/1342-0.txt
Document [1], Score: 0.0271, Link: https://www.gutenberg.org/files/11/11-0.txt

=== Edit_Distance ===
Document [2], Score: 0.0383, Link: https://www.gutenberg.org/files/98/98-0.txt
Document [0], Score: 0.0309, Link: https://www.gutenberg.org/files/1342/1342-0.txt
Document [1], Score: 0.0271, Link: https://www.gutenberg.org/files/11/11-0.txt

=== Soundex Search ===
Document [2], Score: 0.0383, Link: https://www.gutenberg.org/files/98/98-0.txt
Document [0], Score: 0.0309, Link: https://www.gutenberg.org/files/1342/1342-0.txt
Document [1], Score: 0.0271, Link: https://www.gutenberg.org/files/11/11-0.txt

Search: boisterousness

=== Boolean Search ===
Document [0], Score: 0.0006, Link: https://www.gutenberg.org/files/1342/1342-0.txt

=== Edit_Distance ===
Document [0], Score: 0.0006, L